# 01 - Leitura, limpeza e exploração
Dados: PRF - Dados Abertos (https://www.gov.br/prf/pt-br/acesso-a-informacao/dados-abertos/dados-abertos-da-prf).
Os CSVs ficam em `data/` (fora do Git). **Rode célula por célula e confira a saída.**

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = RAIZ / "data"; FIG = RAIZ / "figuras"; FIG.mkdir(exist_ok=True)
ANO = 2023
SEED = 42
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)

## 1. Localizar e ler os arquivos
Separador, decimal e codificação são **NÃO CONFIRMADOS**: a função testa as combinações e mostra o que funcionou.

In [ ]:
print("Arquivos em data/:", [p.name for p in DATA.glob("*")])
def achar(prefixo):
    achados = sorted(DATA.glob(f"{prefixo}{ANO}*.csv"))
    print(prefixo, "->", [a.name for a in achados])
    assert achados, f"Nenhum arquivo {prefixo}{ANO}*.csv em data/. Baixe e descompacte da página da PRF."
    return achados[0]

def ler(path):
    erro = None
    for enc in ("utf-8", "latin-1"):
        for sep in (";", ","):
            try:
                df = pd.read_csv(path, sep=sep, encoding=enc, low_memory=False)
                if df.shape[1] > 5:
                    print(f"OK {path.name}: sep={sep!r} encoding={enc}")
                    return df
            except Exception as e:
                erro = e
    raise erro

oc = ler(achar("datatran"))      # 1 linha por acidente
pe = ler(achar("acidentes"))     # 1 linha por pessoa

In [ ]:
for nome, df in [("OCORRÊNCIA (datatran)", oc), ("PESSOA (acidentes)", pe)]:
    print("="*70); print(nome, df.shape)
    info = pd.DataFrame({"tipo": df.dtypes.astype(str), "nulos": df.isna().sum(), "% nulos": (df.isna().mean()*100).round(2)})
    display(info)
display(oc.head(3)); display(pe.head(3))

## 2. Chave de ligação
Confira nas listas acima o nome da coluna que existe nas duas tabelas (provavelmente `id`, **NÃO CONFIRMADO**).

In [ ]:
comuns = sorted(set(oc.columns) & set(pe.columns))
print("Colunas em comum:", comuns)
CHAVE = "id"   # AJUSTE se a saída acima mostrar outro nome
assert CHAVE in oc.columns and CHAVE in pe.columns, "Chave não encontrada; ajuste CHAVE."
print("Ids únicos em oc:", oc[CHAVE].nunique(), "de", len(oc), "linhas")
print("Ids únicos em pe:", pe[CHAVE].nunique(), "de", len(pe), "linhas")
print("Ids de pe que existem em oc:", pe[CHAVE].isin(oc[CHAVE]).mean().round(4))

## 3. Limpeza (cada decisão está justificada no comentário)

In [ ]:
# Padronizar nomes de colunas (minúsculas, sem espaços)
oc.columns = oc.columns.str.strip().str.lower()
pe.columns = pe.columns.str.strip().str.lower()

# Textos: tirar espaços e padronizar caixa (evita 'Chuva' e 'chuva ' como categorias diferentes)
for df in (oc, pe):
    for col in df.select_dtypes("object").columns:
        df[col] = df[col].astype(str).str.strip().replace({"nan": np.nan, "": np.nan, "NA": np.nan})

# Numéricas com vírgula decimal -> ponto (latitude/longitude/km costumam vir assim)
for col in ["km", "latitude", "longitude"]:
    if col in oc.columns:
        oc[col] = pd.to_numeric(oc[col].astype(str).str.replace(",", ".", regex=False), errors="coerce")

NUM = ["pessoas", "veiculos", "mortos", "feridos_leves", "feridos_graves"]   # NÃO CONFIRMADO: confira
faltam = [c for c in NUM if c not in oc.columns]
assert not faltam, f"Colunas numéricas não encontradas: {faltam}. Veja oc.columns e ajuste NUM."
for c in NUM:
    oc[c] = pd.to_numeric(oc[c], errors="coerce")

n0 = len(oc)
oc = oc.drop_duplicates(subset=CHAVE)                       # 1 linha por acidente
print("Duplicados removidos:", n0 - len(oc))
n1 = len(oc)
oc = oc.dropna(subset=NUM)                                  # sem os números básicos não dá para agrupar
oc = oc[(oc[NUM] >= 0).all(axis=1) & (oc["pessoas"] >= 1) & (oc["veiculos"] >= 1)]  # valores impossíveis
print("Removidos por nulos/valores inválidos:", n1 - len(oc))
print("Linhas finais (ocorrências):", len(oc))

In [ ]:
# Resumo por acidente a partir da tabela de pessoas (só usa colunas que existirem - NÃO CONFIRMADO)
agg = {}
if "idade" in pe.columns:
    pe["idade"] = pd.to_numeric(pe["idade"], errors="coerce").where(lambda s: s.between(0, 110))
    agg["idade_media"] = ("idade", "mean")
if "sexo" in pe.columns:
    agg["pct_masculino"] = ("sexo", lambda s: (s.str.lower().str.startswith("m")).mean())
if "tipo_veiculo" in pe.columns:
    agg["tem_moto"] = ("tipo_veiculo", lambda s: s.str.contains("moto", case=False, na=False).any())
print("Resumos por pessoa disponíveis:", list(agg))
resumo = pe.groupby(CHAVE).agg(**agg).reset_index() if agg else pe[[CHAVE]].drop_duplicates()
base = oc.merge(resumo, on=CHAVE, how="left")
print("Linhas após a junção:", len(base), "| ocorrências sem pessoas na outra tabela:", base[list(agg)[0]].isna().sum() if agg else "n/a")
base.to_pickle(DATA / "base_acidentes.pkl")
print("Salvo em data/base_acidentes.pkl")

## 4. Análise exploratória

In [ ]:
def topo(col, n=10, nome=None):
    if col not in base.columns:
        print(f"[NÃO CONFIRMADO] coluna {col!r} não existe"); return
    plt.figure(figsize=(8, 4))
    ordem = base[col].value_counts().head(n).index
    sns.countplot(data=base, y=col, order=ordem, color="#2b6cb0")
    plt.title(f"Acidentes por {nome or col} ({ANO})"); plt.xlabel("Nº de acidentes"); plt.ylabel("")
    plt.tight_layout(); plt.savefig(FIG / f"eda_{col}.png", dpi=150); plt.show()

for col in ["uf", "causa_acidente", "tipo_acidente", "fase_dia", "condicao_metereologica"]:
    topo(col)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
base["mortos"].clip(upper=5).value_counts().sort_index().plot.bar(ax=ax[0], color="#c53030"); ax[0].set_title("Mortos por acidente (5 = 5 ou mais)")
(base["feridos_leves"] + base["feridos_graves"]).clip(upper=10).value_counts().sort_index().plot.bar(ax=ax[1], color="#dd6b20"); ax[1].set_title("Feridos por acidente (10 = 10 ou mais)")
plt.tight_layout(); plt.savefig(FIG / "eda_mortos_feridos.png", dpi=150); plt.show()
print(base[NUM].describe().round(2))
print("Total de mortos no ano:", int(base["mortos"].sum()), "| acidentes com morte:", int((base["mortos"] > 0).sum()))